# 01 · Sentinel-1 scene inventory

Lists every Sentinel-1 IW GRD (VV+VH) acquisition over Nagaon, Jan–Jul 2022, with pass direction, relative orbit and how much of the district each scene covers. Used to pick the June event dates and a dry-season baseline from **one matching track**.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import ee, geemap, pandas as pd
import config, ee_utils
ee_utils.init()
aoi = ee_utils.district_geometry()
print("District area (km2):", round(aoi.area(maxError=10).divide(1e6).getInfo(), 1))

District area (km2): 2582.4


In [2]:
col = ee_utils.s1_collection(aoi, *config.INVENTORY_WINDOW)
aoi_area = aoi.area(maxError=10)

def describe(img):
    cover = img.geometry().intersection(aoi, maxError=10).area(maxError=10).divide(aoi_area).multiply(100)
    return ee.Feature(None, {
        "id": img.get("system:index"),
        "time": img.date().format("YYYY-MM-dd HH:mm"),
        "platform": img.get("platform_number"),
        "pass": img.get("orbitProperties_pass"),
        "rel_orbit": img.get("relativeOrbitNumber_start"),
        "cover_pct": cover,
    })

inv = pd.DataFrame(ee.FeatureCollection(col.map(describe)).getInfo()["features"]).pipe(
    lambda d: pd.DataFrame(list(d["properties"])))
inv["time"] = pd.to_datetime(inv["time"])
inv["date"] = inv["time"].dt.date
inv = inv.sort_values("time").reset_index(drop=True)
inv

,cover_pct,id,pass,platform,rel_orbit,time,date
0,100.000000,S1A_IW_GRDH_1SDV_20220101T234715_20220101T2347...,DESCENDING,A,77,2022-01-01 23:47:00,2022-01-01
1,55.306683,S1A_IW_GRDH_1SDV_20220106T114859_20220106T1149...,ASCENDING,A,143,2022-01-06 11:48:00,2022-01-06
2,93.201002,S1A_IW_GRDH_1SDV_20220111T115706_20220111T1157...,ASCENDING,A,41,2022-01-11 11:57:00,2022-01-11
3,55.147726,S1A_IW_GRDH_1SDV_20220118T114859_20220118T1149...,ASCENDING,A,143,2022-01-18 11:48:00,2022-01-18
4,93.207951,S1A_IW_GRDH_1SDV_20220123T115706_20220123T1157...,ASCENDING,A,41,2022-01-23 11:57:00,2022-01-23
5,100.000000,S1A_IW_GRDH_1SDV_20220125T234714_20220125T2347...,DESCENDING,A,77,2022-01-25 23:47:00,2022-01-25
6,55.114022,S1A_IW_GRDH_1SDV_20220130T114858_20220130T1149...,ASCENDING,A,143,2022-01-30 11:48:00,2022-01-30
7,93.212892,S1A_IW_GRDH_1SDV_20220204T115705_20220204T1157...,ASCENDING,A,41,2022-02-04 11:57:00,2022-02-04
8,100.000000,S1A_IW_GRDH_1SDV_20220206T234713_20220206T2347...,DESCENDING,A,77,2022-02-06 23:47:00,2022-02-06
9,55.310528,S1A_IW_GRDH_1SDV_20220211T114858_20220211T1149...,ASCENDING,A,143,2022-02-11 11:48:00,2022-02-11


Scenes from the same pass can be split across two adjacent frames on the same date — group by date + track to get the district-level coverage.

In [3]:
by_track = (inv.groupby(["pass", "rel_orbit", "date"])
              .agg(scenes=("id", "count"), cover_pct=("cover_pct", "sum"))
              .reset_index())
by_track["cover_pct"] = by_track["cover_pct"].clip(upper=100).round(1)
out = config.OUT_TABLES / "s1_inventory.csv"
out.parent.mkdir(parents=True, exist_ok=True)
inv.to_csv(out, index=False)
by_track.to_csv(config.OUT_TABLES / "s1_inventory_by_track.csv", index=False)
by_track.pivot_table(index="date", columns=["pass", "rel_orbit"], values="cover_pct")

pass       ASCENDING       DESCENDING
rel_orbit        41    143        77 
date                                 
2022-01-01       NaN   NaN      100.0
2022-01-06       NaN  55.3        NaN
2022-01-11      93.2   NaN        NaN
2022-01-18       NaN  55.1        NaN
2022-01-23      93.2   NaN        NaN
2022-01-25       NaN   NaN      100.0
2022-01-30       NaN  55.1        NaN
2022-02-04      93.2   NaN        NaN
2022-02-06       NaN   NaN      100.0
2022-02-11       NaN  55.3        NaN
2022-02-16      93.2   NaN        NaN
2022-02-18       NaN   NaN      100.0
2022-02-23       NaN  55.4        NaN
2022-02-28      93.2   NaN        NaN
2022-03-02       NaN   NaN      100.0
2022-03-07       NaN  55.4        NaN
2022-03-12      93.2   NaN        NaN
2022-03-14       NaN   NaN      100.0
2022-03-19       NaN  55.4        NaN
2022-03-24      93.2   NaN        NaN
2022-03-26       NaN   NaN      100.0
2022-03-31       NaN  55.2        NaN
2022-04-05      93.2   NaN        NaN
2022-04-07       NaN   NaN      100.0
2022-04-12       NaN  55.1        NaN
2022-04-17      93.2   NaN        NaN
2022-04-19       NaN   NaN      100.0
2022-04-24       NaN  55.3        NaN
2022-04-29      93.2   NaN        NaN
2022-05-01       NaN   NaN      100.0
2022-05-06       NaN  54.9        NaN
2022-05-11      93.2   NaN        NaN
2022-05-13       NaN   NaN      100.0
2022-05-18       NaN  54.9        NaN
2022-05-23      93.2   NaN        NaN
2022-05-25       NaN   NaN      100.0
2022-05-30       NaN  55.2        NaN
2022-06-04      93.2   NaN        NaN
2022-06-06       NaN   NaN      100.0
2022-06-11       NaN  55.3        NaN
2022-06-16      93.2   NaN        NaN
2022-06-18       NaN   NaN      100.0
2022-06-23       NaN  55.2        NaN
2022-06-28      93.2   NaN        NaN
2022-06-30       NaN   NaN      100.0
2022-07-05       NaN  55.3        NaN
2022-07-12       NaN   NaN      100.0
2022-07-17       NaN  55.2        NaN
2022-07-22      93.2   NaN        NaN
2022-07-24       NaN   NaN      100.0
2022-07-29       NaN  55.0        NaN

In [4]:
m = geemap.Map(center=[26.35, 92.75], zoom=9)
m.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Nagaon 2022")
june = col.filterDate(*config.EVENT_WINDOW)
m.addLayer(june.select("VH").min().clip(aoi), {"min": -28, "max": -8}, "June 2022 VH (min)")
m

Map(center=[26.35, 92.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…